# Run the simulation on the cluster (Jupyter + GPU)

Runs Block 4 directly with HuggingFace **transformers** — no server, no Ollama.
It reuses the exact prompts and parsing from `pipeline/simulate.py`, so the output
(`pipeline/sim_out/raw_export.csv` + `logs.jsonl`) is **identical in format** to a
local run and can be cleaned/validated on your laptop with `make clean` /
`pipeline/diagnose.py`.

**Before running:**
1. This repo is cloned on the cluster and this notebook is opened from inside it.
2. You're on a **GPU** runtime.
3. Model weights are downloadable. If compute nodes have **no internet**,
   pre-download on a login node first (`huggingface-cli download <MODEL>`), or set
   `HF_HOME` to a shared cache that already has them.

Then set `MODEL` / `PILOT` in the config cell and **Run All**.

In [ ]:
# 1. Install any missing deps (safe to re-run)
import importlib.util as u, subprocess, sys
need = [m for m in ["torch", "transformers", "accelerate", "tqdm", "requests"]
        if not u.find_spec(m)]
if need:
    print("installing:", need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
print("deps ok")

In [ ]:
# 2. Locate the repo, load the Block-2/3 artifacts and the shared prompt/parse code
import csv, json, time, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "codebook.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "codebook.csv").exists(), f"open this notebook inside the cloned repo; cwd={Path.cwd()}"
PIPE = ROOT / "pipeline"; sys.path.insert(0, str(PIPE))

# reuse the EXACT prompt construction + parsing from the local harness
from simulate import (build_system, build_question_block, build_user,
                      parse_response, _demo_labels, PERSONA_FIELDS)

schema     = json.loads((PIPE / "schema.json").read_text(encoding="utf-8"))
conditions = json.loads((PIPE / "conditions.json").read_text(encoding="utf-8"))
demo       = _demo_labels(schema)
raw_cols   = schema["raw_export_columns"]
question_block, spec = build_question_block(schema)
stim       = {lab: e["stimulus_text"] for lab, e in conditions["interventions"].items()}
personas   = list(csv.DictReader((PIPE / "personas.csv").open(encoding="utf-8")))
print(f"{len(personas)} personas; {len(spec)} items per respondent")

In [ ]:
# 3. Run settings  --  edit these
MODEL       = "Qwen/Qwen2.5-7B-Instruct"   # ungated, no token needed.
                                           # For Llama (matches local): "meta-llama/Llama-3.1-8B-Instruct"
                                           # -> needs `huggingface-cli login` + license acceptance.
TEMPERATURE = 1.0                          # see the literature discussion (0.7 = Argyle precedent, 1.0 = more spread)
PILOT       = True                         # True = 1 respondent per condition (17); set False for the full run
LIMIT       = None                         # or an int cap when PILOT is False
OUT         = PIPE / "sim_out"; OUT.mkdir(exist_ok=True)
print(f"MODEL={MODEL}  TEMPERATURE={TEMPERATURE}  PILOT={PILOT}")

In [ ]:
# 4. Load the model onto the GPU
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print("loading", MODEL, "...")
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
print("loaded on", model.device)

In [ ]:
# 5. Generation helper (chat template -> sample -> text). JSON is requested in the prompt.
def extract_json(t):
    i, j = t.find("{"), t.rfind("}")
    return t[i:j + 1] if (i != -1 and j != -1 and j > i) else t

@torch.no_grad()
def generate_one(messages, seed):
    torch.manual_seed(seed)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=1200, do_sample=True,
                         temperature=TEMPERATURE, top_p=0.95, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

In [ ]:
# 6. Run  (resumable: re-run to continue; tqdm shows progress)
from tqdm.auto import tqdm

if PILOT:
    seen, selected = set(), []
    for p in personas:
        if p["condition"] not in seen:
            seen.add(p["condition"]); selected.append(p)
else:
    selected = personas[:LIMIT] if LIMIT else personas

out_csv, log_path = OUT / "raw_export.csv", OUT / "logs.jsonl"
done = {r["profile_id"] for r in csv.DictReader(out_csv.open(encoding="utf-8"))} if out_csv.exists() else set()
todo = [p for p in selected if p["profile_id"] not in done]
print(f"{len(selected)} selected | {len(done)} already done | {len(todo)} to run")

csv_f = out_csv.open("a", newline="", encoding="utf-8")
writer = csv.DictWriter(csv_f, fieldnames=raw_cols)
if not done:
    writer.writeheader()
log_f = log_path.open("a", encoding="utf-8")

n_issue = 0
for p in tqdm(todo, desc="simulating", unit="resp"):
    cond = p["condition"]
    seed = int("".join(c for c in p["profile_id"] if c.isdigit()) or "0")
    messages = [{"role": "system", "content": build_system(p, demo)},
                {"role": "user", "content": build_user(cond, stim.get(cond), question_block)}]
    raw, values, issues = "", {}, ["no_attempt"]
    for attempt in range(3):
        try:
            raw = generate_one(messages, seed + attempt)
            values, issues = parse_response(extract_json(raw), spec)
            if not any(i.startswith("json_error") for i in issues) and \
               sum(i.startswith("missing") for i in issues) <= 5:
                break
        except Exception as e:
            issues = [f"error:{e}"]
    row = {c: "" for c in raw_cols}
    for fld in PERSONA_FIELDS:
        row[fld] = p[fld]
    row.update({k: v for k, v in values.items() if k in row})
    writer.writerow(row); csv_f.flush()
    log_f.write(json.dumps({"profile_id": p["profile_id"], "condition": cond, "model": MODEL,
                            "seed": seed, "messages": messages, "raw_response": raw,
                            "parse_issues": issues, "ts": time.time()}) + "\n"); log_f.flush()
    if issues:
        n_issue += 1
csv_f.close(); log_f.close()
print(f"\ndone -> {out_csv}  (+ {log_path})")
print(f"respondents with parse issues: {n_issue}/{len(todo)}")

In [ ]:
# 7. Quick peek
rows = list(csv.DictReader(out_csv.open(encoding="utf-8")))
print("rows:", len(rows), "| columns match schema:", list(rows[0]) == raw_cols)
print("conditions covered:", len({r["condition"] for r in rows}), "/ 17")

## Get your results back

In the Jupyter file browser, download **`pipeline/sim_out/raw_export.csv`** and
**`pipeline/sim_out/logs.jsonl`** (right-click → Download). On your **local**
machine, drop them into `pipeline/sim_out/` in the repo and run:

```bash
python3 pipeline/diagnose.py            # spread / skew report (needs local R)
```

When you're happy with a full run, those two files become the submission inputs
(Block 5/6): the raw export goes to `raw_data_deposit/`, then `make clean` +
`make check`.

> This transformers path is the simplest "just run it" option. For a much faster
> large run, a vLLM server + `pipeline/simulate.py --backend openai` batches
> generation — ask and I'll add a one-cell version.